In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/sraddhasigdel/datasetmciad/Multimodalmciad/csv.csv
/kaggle/input/datasets/sraddhasigdel/datasetmciad/Multimodalmciad/PET/0/027_S_4919.nii
/kaggle/input/datasets/sraddhasigdel/datasetmciad/Multimodalmciad/PET/0/041_S_4874.nii
/kaggle/input/datasets/sraddhasigdel/datasetmciad/Multimodalmciad/PET/0/005_S_4185.nii
/kaggle/input/datasets/sraddhasigdel/datasetmciad/Multimodalmciad/PET/0/032_S_0214.nii
/kaggle/input/datasets/sraddhasigdel/datasetmciad/Multimodalmciad/PET/0/027_S_0835.nii
/kaggle/input/datasets/sraddhasigdel/datasetmciad/Multimodalmciad/PET/0/127_S_0925.nii
/kaggle/input/datasets/sraddhasigdel/datasetmciad/Multimodalmciad/PET/0/126_S_1077.nii
/kaggle/input/datasets/sraddhasigdel/datasetmciad/Multimodalmciad/PET/0/114_S_1106.nii
/kaggle/input/datasets/sraddhasigdel/datasetmciad/Multimodalmciad/PET/0/073_S_1357.nii
/kaggle/input/datasets/sraddhasigdel/datasetmciad/Multimodalmciad/PET/0/123_S_4127.nii
/kaggle/input/datasets/sraddhasigdel/datasetmciad/Multim

In [2]:
data_root = "/kaggle/input/datasets/sraddhasigdel/datasetmciad/Multimodalmciad"
csv       = "/kaggle/input/datasets/sraddhasigdel/datasetmciad/Multimodalmciad/csv.csv"
mri_root  = "/kaggle/input/datasets/sraddhasigdel/datasetmciad/Multimodalmciad/MRI"
pet_root  = "/kaggle/input/datasets/sraddhasigdel/datasetmciad/Multimodalmciad/PET"

In [3]:
"""
dataset.py — MultiModal Alzheimer's Dataset Loader
Handles 3D MRI + PET NIfTI volumes + tabular clinical features.

Modality-specific preprocessing assumptions:
  MRI : skull-stripped, bias-corrected, standard-space, min-max [0, 1]
  PET : SUVR-normalised only (NOT min-max) — raw SUVR ratios ~0.5–2.5+
        → values are NOT bounded to [0, 1]; do not clamp to 1.0
"""

import os
import glob
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset
import nibabel as nib
from scipy.ndimage import zoom


# ─────────────────────────────────────────────
#  Shared helper: load raw volume + resample
# ─────────────────────────────────────────────
def _load_and_resample(path: str, target_shape=(96, 96, 96)) -> np.ndarray:
    """Load NIfTI, strip 4th dim if present, resample only if needed."""
    img = nib.load(path)
    vol = img.get_fdata(dtype=np.float32)

    if vol.ndim == 4:
        vol = vol[..., 0]

    current_shape = vol.shape[:3]
    if current_shape != tuple(target_shape):
        factors = [t / s for t, s in zip(target_shape, current_shape)]
        vol = zoom(vol, factors, order=1)   # trilinear interpolation

    return vol.astype(np.float32)


def load_mri(path: str, target_shape=(96, 96, 96)) -> np.ndarray:
    """
    Load MRI volume: skull-stripped, bias-corrected, min-max [0, 1].

    A clamp to [0, 1] is safe here because min-max normalisation
    guarantees the range; the clamp only absorbs sub-voxel fp drift
    introduced during trilinear resampling.
    """
    vol = _load_and_resample(path, target_shape)
    vol = np.clip(vol, 0.0, 1.0)   # absorb fp drift, range is known [0,1]
    return vol


def load_pet(path: str, target_shape=(96, 96, 96)) -> np.ndarray:
    """
    Load PET volume: SUVR-normalised only — NOT min-max normalised.

    SUVR = uptake(voxel) / mean_uptake(reference_region)

    Typical ranges:
      FDG-PET  : ~0.5 – 2.0  (hypometabolism in AD regions drops toward 0.5)
      Amyloid  : ~1.0 – 3.0+ (elevated in amyloid-positive subjects)

    Do NOT clamp to [0, 1] — that would truncate all voxels with
    SUVR > 1.0, destroying the primary diagnostic signal.
    Only clip at 0 to remove any negative fp artefacts from resampling.
    """
    vol = _load_and_resample(path, target_shape)
    vol = np.clip(vol, 0.0, None)   # only remove negative resampling artefacts
    return vol


# ─────────────────────────────────────────────
#  Tabular feature columns
# ─────────────────────────────────────────────
#
# EXCLUDED (clinical diagnosis scores — circular with label in ADNI):
#   CDRSB, MMSE, ADAS11, ADAS13, FAQ
#   RAVLT_immediate, RAVLT_learning, RAVLT_forgetting, RAVLT_perc_forgetting
#
#   These scores are used by ADNI as PRIMARY CRITERIA to assign CN/MCI/AD
#   labels. Including them as features means predicting the label from the
#   variables that define it — a logistic regression achieves AUC=1.0
#   on them alone, so they contribute no meaningful signal beyond trivially
#   solving the task before any imaging is considered.
#
# KEPT (biological markers + demographics — independent of diagnosis label):
TABULAR_COLS = [
    # Demographics — confounders, not diagnosis criteria
    "AGE", "PTGENDER", "PTEDUCAT",
    # Genetic risk
    "APOE4",
    # CSF biomarkers — biological Alzheimer's pathology markers
    "ABETA", "TAU", "PTAU",
    # Derived biomarker ratios — capture amyloid/tau relationship
    "ABETA_TAU_ratio", "ABETA_PTAU_ratio", "TAU_PTAU_ratio",
]


# ─────────────────────────────────────────────
#  Dataset
# ─────────────────────────────────────────────
class AlzheimerMultimodalDataset(Dataset):
    """
    Expects:
        dataset/
          mri/{0,1}/<PTID>.nii[.gz]
          pet/{0,1}/<PTID>.nii[.gz]
          clinical.csv  (columns: PTID, AGE, …, label)
    """

    def __init__(
        self,
        csv_path: str,
        mri_root: str,
        pet_root: str,
        vol_shape: tuple = (96, 96, 96),
        augment: bool = False,
        tabular_cols: list = TABULAR_COLS,
    ):
        self.df = pd.read_csv(csv_path)
        self.mri_root = mri_root
        self.pet_root = pet_root
        self.vol_shape = vol_shape
        self.augment = augment
        self.tabular_cols = tabular_cols

        # Impute & scale tabular features once
        self._prepare_tabular()

        # Build path maps: PTID → file
        self.mri_map = self._build_path_map(mri_root)
        self.pet_map = self._build_path_map(pet_root)

    # ── helpers ───────────────────────────────
    def _build_path_map(self, root: str) -> dict:
        mapping = {}
        for label_dir in ["0", "1"]:
            folder = os.path.join(root, label_dir)
            for fp in glob.glob(os.path.join(folder, "*.nii*")):
                ptid = os.path.basename(fp).split(".")[0]
                mapping[ptid] = fp
        return mapping

    def _prepare_tabular(self, fit_stats: bool = True):
        """
        Encode, impute, and optionally fit normalisation statistics.

        fit_stats=True  : compute mean/std from THIS split (use for train).
        fit_stats=False : mean/std must be set externally before __getitem__
                          is called (use for val — avoids leakage).
        """
        # Encode gender
        if "PTGENDER" in self.df.columns:
            self.df["PTGENDER"] = (self.df["PTGENDER"]
                                   .map({"Male": 0, "Female": 1, 1: 0, 2: 1})
                                   .fillna(0))
        # Median imputation (using this split's medians only)
        for c in self.tabular_cols:
            if c in self.df.columns:
                self.df[c] = self.df[c].fillna(self.df[c].median())

        # Fit normalisation stats on this split only
        if fit_stats:
            self.tab_mean = self.df[self.tabular_cols].mean().values.astype(np.float32)
            self.tab_std  = self.df[self.tabular_cols].std().values.astype(np.float32) + 1e-8
        else:
            # Placeholder — must be overwritten before use
            self.tab_mean = None
            self.tab_std  = None

    def _augment_mri(self, vol: np.ndarray) -> np.ndarray:
        """
        Augmentation for MRI: skull-stripped, bias-corrected, min-max [0,1].

        Bias field correction (e.g. ANTs N4) removes low-frequency coil
        non-uniformity, so the residual intensity variation across scanners
        is small. A mild multiplicative scale U(0.95, 1.05) is still valid
        to simulate residual between-site gain differences after correction.

        Operations:
          1. Random axis flip (p=0.5)  -- orientation equivariance.
          2. Intensity scaling x U(0.95, 1.05) -- residual scanner gain.
          3. Additive Gaussian noise N(0, 0.01) -- scanner thermal noise.
        """
        for ax in range(3):
            if np.random.rand() < 0.5:
                vol = np.flip(vol, axis=ax).copy()

        vol = vol * np.random.uniform(0.95, 1.05)
        vol += np.random.randn(*vol.shape).astype(np.float32) * 0.01
        return np.clip(vol, 0.0, 1.0)

    def _augment_pet(self, vol: np.ndarray) -> np.ndarray:
        """
        Augmentation for PET: SUVR-normalised only (raw ratios, NOT [0,1]).

        SUVR (Standardised Uptake Value Ratio):
            SUVR(v) = uptake(v) / mean_uptake(reference_region)

        Values are physically meaningful ratios with NO fixed upper bound.
        Typical range: ~0.5 – 2.5 for FDG-PET; higher for amyloid PET.

        Multiplicative intensity scaling is EXCLUDED -- it would shift
        all SUVR values by a constant factor, altering the ratio between
        regions relative to the reference region and corrupting the
        normalisation that SUVR was designed to establish.

        Operations:
          1. Random axis flip (p=0.5) -- orientation equivariance.
          2. Additive Gaussian noise N(0, sigma) where sigma = 1% of the
             volume's own standard deviation -- this is SUVR-scale-aware,
             so noise magnitude is always proportional to the actual signal
             range rather than a hardcoded absolute value.

        Clamp: only at 0 to remove negative resampling artefacts.
        No upper clamp -- SUVR has no fixed ceiling.
        """
        for ax in range(3):
            if np.random.rand() < 0.5:
                vol = np.flip(vol, axis=ax).copy()

        # Scale-aware noise: 1% of this volume's std (SUVR-range-safe)
        sigma = vol.std() * 0.01
        vol  += np.random.randn(*vol.shape).astype(np.float32) * sigma
        return np.clip(vol, 0.0, None)   # no upper clamp for SUVR

    # ── protocol ──────────────────────────────
    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        ptid  = str(row["PTID"])
        label = int(row["label"])

        # ── 3-D volumes ───────────────────────
        mri_vol = load_mri(self.mri_map[ptid], self.vol_shape)
        pet_vol = load_pet(self.pet_map[ptid], self.vol_shape)

        if self.augment:
            mri_vol = self._augment_mri(mri_vol)   # bias-corrected MRI
            pet_vol = self._augment_pet(pet_vol)   # SUVR-normalised PET

        # Add channel dim → (1, D, H, W)
        mri_tensor = torch.from_numpy(mri_vol).unsqueeze(0)
        pet_tensor = torch.from_numpy(pet_vol).unsqueeze(0)

        # ── Tabular ───────────────────────────
        tab_raw = row[self.tabular_cols].values.astype(np.float32)
        tab_norm = (tab_raw - self.tab_mean) / self.tab_std
        tab_tensor = torch.from_numpy(tab_norm)

        return {
            "mri":   mri_tensor,          # (1, 96, 96, 96)
            "pet":   pet_tensor,          # (1, 96, 96, 96)
            "tab":   tab_tensor,          # (19,)
            "label": torch.tensor(label, dtype=torch.long),
            "ptid":  ptid,
        }

In [4]:
"""
encoders.py — 3D ResNet Encoders for MRI & PET + Tabular MLP Encoder
"""

import torch
import torch.nn as nn
import torch.nn.functional as F


# ══════════════════════════════════════════════════════════════════════
#  3-D ResNet Primitives
# ══════════════════════════════════════════════════════════════════════

class Conv3dBnReLU(nn.Sequential):
    def __init__(self, in_ch, out_ch, kernel=3, stride=1, padding=1):
        super().__init__(
            nn.Conv3d(in_ch, out_ch, kernel, stride, padding, bias=False),
            nn.BatchNorm3d(out_ch),
            nn.ReLU(inplace=True),
        )


class ResBlock3D(nn.Module):
    """
    Pre-activation 3-D residual block with optional squeeze-and-excitation.

    Math:
        y = x + SE( W2 · ReLU( BN( W1 · BN(x) ) ) )
        SE(z) = z ⊙ σ( W_e2 · ReLU( W_e1 · GAP(z) ) )   [channel attention]
    """

    def __init__(self, in_ch, out_ch, stride=1, use_se=True, reduction=8):
        super().__init__()
        self.conv1 = nn.Conv3d(in_ch, out_ch, 3, stride, 1, bias=False)
        self.bn1   = nn.BatchNorm3d(out_ch)
        self.conv2 = nn.Conv3d(out_ch, out_ch, 3, 1, 1, bias=False)
        self.bn2   = nn.BatchNorm3d(out_ch)
        self.relu  = nn.ReLU(inplace=True)

        # Squeeze-and-Excitation
        self.use_se = use_se
        if use_se:
            mid = max(out_ch // reduction, 4)
            self.se = nn.Sequential(
                nn.AdaptiveAvgPool3d(1),
                nn.Flatten(),
                nn.Linear(out_ch, mid, bias=False),
                nn.ReLU(inplace=True),
                nn.Linear(mid, out_ch, bias=False),
                nn.Sigmoid(),
            )

        # Shortcut projection when dims change
        self.shortcut = nn.Sequential()
        if stride != 1 or in_ch != out_ch:
            self.shortcut = nn.Sequential(
                nn.Conv3d(in_ch, out_ch, 1, stride, bias=False),
                nn.BatchNorm3d(out_ch),
            )

    def forward(self, x):
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        if self.use_se:
            scale = self.se(out).view(out.size(0), out.size(1), 1, 1, 1)
            out   = out * scale
        return self.relu(out + self.shortcut(x))


# ══════════════════════════════════════════════════════════════════════
#  3-D ResNet-10 Backbone  (lightweight, ideal for N=400)
# ══════════════════════════════════════════════════════════════════════

class ResNet3D(nn.Module):
    """
    ResNet-10 variant for volumetric input (1, D, H, W).
    Latent dim = embed_dim.

    Architecture:
        Stem  : Conv3d(1→32) → BN → ReLU → MaxPool3d
        Layer1: [32→64,  stride=2]
        Layer2: [64→128, stride=2]
        Layer3: [128→256,stride=2]
        GAP   → Linear(256→embed_dim)
    """

    def __init__(self, in_channels=1, embed_dim=256, use_se=True, dropout=0.3):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv3d(in_channels, 32, 7, stride=2, padding=3, bias=False),
            nn.BatchNorm3d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool3d(3, stride=2, padding=1),
        )
        self.layer1 = ResBlock3D(32,  64,  stride=2, use_se=use_se)
        self.layer2 = ResBlock3D(64,  128, stride=2, use_se=use_se)
        self.layer3 = ResBlock3D(128, 256, stride=2, use_se=use_se)

        self.gap     = nn.AdaptiveAvgPool3d(1)
        self.dropout = nn.Dropout(dropout)
        self.proj    = nn.Linear(256, embed_dim)

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv3d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
            elif isinstance(m, nn.BatchNorm3d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)

    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.gap(x).flatten(1)   # (B, 256)
        x = self.dropout(x)
        return self.proj(x)           # (B, embed_dim)


# ══════════════════════════════════════════════════════════════════════
#  Tabular MLP Encoder
# ══════════════════════════════════════════════════════════════════════

class TabularEncoder(nn.Module):
    """
    MLP with residual connections for clinical / biomarker features.

    Math (one residual block):
        h1 = GELU( LN( W1·x + b1 ) )
        h2 = Dropout( GELU( LN( W2·h1 + b2 ) ) )
        out = h2 + Linear_proj(x)   if dim changes, else h2 + x
    """

    def __init__(self, in_dim: int, hidden_dim: int = 128, embed_dim: int = 256, dropout: float = 0.3):
        super().__init__()
        self.block1 = self._res_block(in_dim,     hidden_dim, dropout)
        self.block2 = self._res_block(hidden_dim, hidden_dim, dropout)
        self.proj   = nn.Linear(hidden_dim, embed_dim)

        # Shortcut projections for residuals when dims change
        self.sc1 = nn.Linear(in_dim,     hidden_dim, bias=False)
        self.sc2 = nn.Identity()

    @staticmethod
    def _res_block(in_d, out_d, drop):
        return nn.Sequential(
            nn.Linear(in_d, out_d),
            nn.LayerNorm(out_d),
            nn.GELU(),
            nn.Dropout(drop),
            nn.Linear(out_d, out_d),
            nn.LayerNorm(out_d),
            nn.GELU(),
        )

    def forward(self, x):
        # Block 1 with residual
        h = self.block1(x) + self.sc1(x)
        # Block 2 with residual
        h = self.block2(h) + h
        return self.proj(h)            # (B, embed_dim)

In [5]:
"""
fusion.py — Intermediate Fusion Modules
  1. Cross-Modal Attention Fusion
  2. Gated Multimodal Unit (GMU)
  3. Full Trimodal Fusion (MRI + PET + Tabular)
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
import math


# ══════════════════════════════════════════════════════════════════════
#  1.  Cross-Modal Attention (Transformer-style)
# ══════════════════════════════════════════════════════════════════════

class CrossModalAttention(nn.Module):
    """
    Scaled Dot-Product Cross-Attention between two modality embeddings.

    Given query Q from modality A and key/value K,V from modality B:

        Attention(Q, K, V) = softmax( QK^T / √d_k ) · V

    where Q = x_A · W_Q ,  K = x_B · W_K ,  V = x_B · W_V
    Output is modality-A enriched with context from modality-B.
    """

    def __init__(self, embed_dim: int, num_heads: int = 4, dropout: float = 0.1):
        super().__init__()
        assert embed_dim % num_heads == 0
        self.h    = num_heads
        self.d_k  = embed_dim // num_heads

        self.W_Q  = nn.Linear(embed_dim, embed_dim, bias=False)
        self.W_K  = nn.Linear(embed_dim, embed_dim, bias=False)
        self.W_V  = nn.Linear(embed_dim, embed_dim, bias=False)
        self.out  = nn.Linear(embed_dim, embed_dim)
        self.drop = nn.Dropout(dropout)
        self.norm = nn.LayerNorm(embed_dim)

    def forward(self, query: torch.Tensor, context: torch.Tensor) -> torch.Tensor:
        """
        query   : (B, D)   — the modality that wants to attend
        context : (B, D)   — the modality being attended to
        """
        B = query.size(0)
        # Expand to sequence length = 1 for single-vector embeddings
        Q = self.W_Q(query).view(B, 1, self.h, self.d_k).transpose(1, 2)   # (B,h,1,d_k)
        K = self.W_K(context).view(B, 1, self.h, self.d_k).transpose(1, 2)
        V = self.W_V(context).view(B, 1, self.h, self.d_k).transpose(1, 2)

        # Scaled dot-product
        scale  = math.sqrt(self.d_k)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / scale  # (B,h,1,1)
        attn   = self.drop(F.softmax(scores, dim=-1))
        out    = torch.matmul(attn, V)                          # (B,h,1,d_k)

        out = out.transpose(1, 2).contiguous().view(B, -1)     # (B, embed_dim)
        out = self.out(out)

        # Residual + LayerNorm
        return self.norm(out + query)


# ══════════════════════════════════════════════════════════════════════
#  2.  Gated Multimodal Unit (GMU)
# ══════════════════════════════════════════════════════════════════════

class GatedMultimodalUnit(nn.Module):
    """
    GMU learns a soft gate over each modality representation.

    z_m = tanh( W_m · x_m )   for m ∈ {mri, pet, tab}
    h_m = tanh( U_m · x_m )
    g   = softmax( [z_mri; z_pet; z_tab] )   (gate across modalities)
    out = Σ_m  g_m ⊙ h_m

    Reference: Arevalo et al., "Gated Multimodal Units for Information Fusion", 2020
    """

    def __init__(self, embed_dim: int, n_modalities: int = 3, dropout: float = 0.1):
        super().__init__()
        self.n = n_modalities
        self.gate_layers = nn.ModuleList(
            [nn.Linear(embed_dim, embed_dim) for _ in range(n_modalities)]
        )
        self.feat_layers = nn.ModuleList(
            [nn.Linear(embed_dim, embed_dim) for _ in range(n_modalities)]
        )
        self.gate_norm = nn.Linear(embed_dim * n_modalities, n_modalities)
        self.drop      = nn.Dropout(dropout)
        self.norm      = nn.LayerNorm(embed_dim)

    def forward(self, *embeddings) -> torch.Tensor:
        """
        embeddings: tuple of (B, embed_dim) tensors, one per modality
        """
        assert len(embeddings) == self.n
        z = [torch.tanh(self.gate_layers[i](e)) for i, e in enumerate(embeddings)]
        h = [torch.tanh(self.feat_layers[i](e)) for i, e in enumerate(embeddings)]

        # Gate weights across modalities (soft selection)
        gate_input = torch.cat(z, dim=-1)            # (B, D*n)
        gates      = F.softmax(self.gate_norm(gate_input), dim=-1)  # (B, n)

        # Weighted sum
        out = sum(gates[:, i:i+1] * h[i] for i in range(self.n))    # (B, D)
        return self.norm(self.drop(out))


# ══════════════════════════════════════════════════════════════════════
#  3.  Full Trimodal Intermediate Fusion Block
# ══════════════════════════════════════════════════════════════════════

class TrimodalFusion(nn.Module):
    """
    Intermediate Fusion Pipeline:

    Step 1 — Bidirectional Cross-Modal Attention
        z_mri'  = CMA(Q=mri,  K/V=pet)
        z_pet'  = CMA(Q=pet,  K/V=mri)
        z_mri'' = CMA(Q=mri', K/V=tab)   [image ← biomarker context]
        z_pet'' = CMA(Q=pet', K/V=tab)

    Step 2 — Gated Fusion over {z_mri'', z_pet'', z_tab}
        z_fused = GMU(z_mri'', z_pet'', z_tab)

    Step 3 — Projection to classifier input dimension
        out = LayerNorm( GELU( W · z_fused ) )
    """

    def __init__(self, embed_dim: int = 256, num_heads: int = 4,
                 out_dim: int = 512, dropout: float = 0.1):
        super().__init__()
        # Cross-modal attention pairs
        self.mri_pet = CrossModalAttention(embed_dim, num_heads, dropout)
        self.pet_mri = CrossModalAttention(embed_dim, num_heads, dropout)
        self.mri_tab = CrossModalAttention(embed_dim, num_heads, dropout)
        self.pet_tab = CrossModalAttention(embed_dim, num_heads, dropout)

        # Gated fusion
        self.gmu = GatedMultimodalUnit(embed_dim, n_modalities=3, dropout=dropout)

        # Output projection
        self.proj = nn.Sequential(
            nn.Linear(embed_dim, out_dim),
            nn.LayerNorm(out_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )

    def forward(self, z_mri: torch.Tensor, z_pet: torch.Tensor,
                z_tab: torch.Tensor) -> torch.Tensor:
        # Step 1: cross-attend
        z_mri_p  = self.mri_pet(z_mri, z_pet)
        z_pet_p  = self.pet_mri(z_pet, z_mri)
        z_mri_pp = self.mri_tab(z_mri_p, z_tab)
        z_pet_pp = self.pet_tab(z_pet_p, z_tab)

        # Step 2: gated fusion
        z_fused  = self.gmu(z_mri_pp, z_pet_pp, z_tab)

        # Step 3: project
        return self.proj(z_fused)      # (B, out_dim)

In [6]:
"""
model.py — AlzheimerMultimodalNet
Full pipeline: encode → intermediate fusion → classify
"""

import torch
import torch.nn as nn
import torch.nn.functional as F


# ══════════════════════════════════════════════════════════════════════
#  Classification Head
# ══════════════════════════════════════════════════════════════════════

class ClassificationHead(nn.Module):
    """
    Multi-layer classifier with Label Smoothing support.

    Architecture:
        Linear(fused_dim → 256) → GELU → Dropout
        Linear(256 → 128)       → GELU → Dropout
        Linear(128 → num_classes)

    The deep head adds non-linear capacity while Dropout (p=0.4) combats
    overfitting on the small N=400 cohort.
    """

    def __init__(self, in_dim: int, num_classes: int = 2, dropout: float = 0.4):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 256),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(256, 128),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        return self.net(x)


# ══════════════════════════════════════════════════════════════════════
#  Full Model
# ══════════════════════════════════════════════════════════════════════

class AlzheimerMultimodalNet(nn.Module):
    """
    Trimodal Alzheimer's Staging Network.

    Modalities:
        • MRI  : 3-D T1-weighted volume   → ResNet3D(SE) → z_mri  ∈ R^D
        • PET  : 3-D FDG-PET volume       → ResNet3D(SE) → z_pet  ∈ R^D
        • Tab  : clinical / biomarker CSV  → TabularMLP   → z_tab  ∈ R^D

    Fusion:
        TrimodalFusion: Cross-Modal Attention × 4 → GMU → proj ∈ R^{2D}

    Classifier:
        MLP(2D → 256 → 128 → num_classes)

    Args:
        tab_in_dim  : number of tabular features (default 19)
        embed_dim   : latent dimension per modality (default 256)
        fused_dim   : output of fusion module (default 512)
        num_classes : 2 for binary staging
        dropout     : global dropout rate
    """

    def __init__(
        self,
        tab_in_dim:  int = 10,   # AGE, PTGENDER, PTEDUCAT, APOE4, ABETA, TAU, PTAU, 3 ratios
        embed_dim:   int = 256,
        fused_dim:   int = 512,
        num_classes: int = 2,
        dropout:     float = 0.3,
        use_se:      bool = True,
    ):
        super().__init__()

        # ── Encoders ──────────────────────────────────────────────────
        self.mri_encoder = ResNet3D(
            in_channels=1, embed_dim=embed_dim, use_se=use_se, dropout=dropout
        )
        self.pet_encoder = ResNet3D(
            in_channels=1, embed_dim=embed_dim, use_se=use_se, dropout=dropout
        )
        self.tab_encoder = TabularEncoder(
            in_dim=tab_in_dim, hidden_dim=128, embed_dim=embed_dim, dropout=dropout
        )

        # ── Fusion ────────────────────────────────────────────────────
        self.fusion = TrimodalFusion(
            embed_dim=embed_dim, num_heads=4, out_dim=fused_dim, dropout=dropout
        )

        # ── Classifier ────────────────────────────────────────────────
        self.classifier = ClassificationHead(
            in_dim=fused_dim, num_classes=num_classes, dropout=dropout + 0.1
        )

        # ── Auxiliary unimodal heads (for deep supervision) ───────────
        self.aux_mri = nn.Linear(embed_dim, num_classes)
        self.aux_pet = nn.Linear(embed_dim, num_classes)
        self.aux_tab = nn.Linear(embed_dim, num_classes)

    def forward(
        self,
        mri:        torch.Tensor,          # (B, 1, D, H, W)
        pet:        torch.Tensor,          # (B, 1, D, H, W)
        tab:        torch.Tensor,          # (B, tab_in_dim)
        return_aux: bool = False,
    ):
        # ── Encode ────────────────────────────────────────────────────
        z_mri = self.mri_encoder(mri)       # (B, embed_dim)
        z_pet = self.pet_encoder(pet)       # (B, embed_dim)
        z_tab = self.tab_encoder(tab)       # (B, embed_dim)

        # ── Fuse ──────────────────────────────────────────────────────
        z_fused = self.fusion(z_mri, z_pet, z_tab)   # (B, fused_dim)

        # ── Classify ──────────────────────────────────────────────────
        logits = self.classifier(z_fused)   # (B, num_classes)

        if return_aux:
            return logits, {
                "mri": self.aux_mri(z_mri),
                "pet": self.aux_pet(z_pet),
                "tab": self.aux_tab(z_tab),
            }
        return logits

    # ── convenience ───────────────────────────────────────────────────
    def encode_all(self, mri, pet, tab):
        """Return individual latent vectors (useful for analysis)."""
        return (
            self.mri_encoder(mri),
            self.pet_encoder(pet),
            self.tab_encoder(tab),
        )

    def count_parameters(self) -> int:
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

In [7]:
"""
train.py — Training + Evaluation Pipeline
Supports: Focal Loss, Label Smoothing, Deep Supervision,
          Cosine-Warmup LR, Stratified K-Fold CV
"""
import os
import math
import time
import random
import argparse
import numpy as np
import pandas as pd
from pathlib import Path
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torch.amp import GradScaler, autocast                             # CHANGED
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    roc_auc_score, balanced_accuracy_score,
    classification_report, confusion_matrix,
)
# ══════════════════════════════════════════════════════════════════════
#  Reproducibility
# ══════════════════════════════════════════════════════════════════════
def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
# ══════════════════════════════════════════════════════════════════════
#  Loss Functions
# ══════════════════════════════════════════════════════════════════════
class FocalLoss(nn.Module):
    def __init__(self, gamma: float = 2.0, alpha: float = 0.25,
                 label_smoothing: float = 0.1):
        super().__init__()
        self.gamma   = gamma
        self.alpha   = alpha
        self.smooth  = label_smoothing
    def forward(self, logits, targets):
        n_cls = logits.size(-1)
        with torch.no_grad():
            smooth_targets = torch.zeros_like(logits).scatter_(
                1, targets.unsqueeze(1), 1.0
            )
            smooth_targets = smooth_targets * (1 - self.smooth) + self.smooth / n_cls
        log_p  = F.log_softmax(logits, dim=-1)
        p      = torch.exp(log_p)
        p_t    = (smooth_targets * p).sum(dim=-1)
        alpha_t = torch.where(targets == 1,
                              torch.tensor(self.alpha, device=logits.device),
                              torch.tensor(1 - self.alpha, device=logits.device))
        loss = -alpha_t * (1 - p_t) ** self.gamma * (smooth_targets * log_p).sum(-1)
        return loss.mean()
class MultimodalLoss(nn.Module):
    def __init__(self, aux_weight: float = 0.3, **focal_kwargs):
        super().__init__()
        self.aux_w    = aux_weight
        self.focal    = FocalLoss(**focal_kwargs)
    def forward(self, logits, aux_logits: dict, targets):
        loss_main = self.focal(logits, targets)
        loss_aux  = sum(self.focal(v, targets) for v in aux_logits.values()) / 3
        return loss_main + self.aux_w * loss_aux, {
            "main": loss_main.item(), "aux": loss_aux.item()
        }
# ══════════════════════════════════════════════════════════════════════
#  LR Scheduler: Cosine with Linear Warm-Up
# ══════════════════════════════════════════════════════════════════════
class CosineWarmupScheduler:
    def __init__(self, optimizer, warmup_steps, total_steps,
                 eta_min=1e-6, eta_max=3e-4):
        self.opt    = optimizer
        self.warmup = warmup_steps
        self.total  = total_steps
        self.eta_min = eta_min
        self.eta_max = eta_max
        self.step_num = 0
    def step(self):
        self.step_num += 1
        t = self.step_num
        if t < self.warmup:
            lr = self.eta_max * t / self.warmup
        else:
            progress = (t - self.warmup) / max(1, self.total - self.warmup)
            lr = self.eta_min + 0.5 * (self.eta_max - self.eta_min) * (
                1 + math.cos(math.pi * progress)
            )
        for pg in self.opt.param_groups:
            pg["lr"] = lr
        return lr
# ══════════════════════════════════════════════════════════════════════
#  Metrics
# ══════════════════════════════════════════════════════════════════════
@torch.no_grad()
def evaluate(model, loader, device, criterion=None):
    model.eval()
    all_logits, all_labels = [], []
    total_loss = 0.0
    for batch in loader:
        mri   = batch["mri"].to(device)
        pet   = batch["pet"].to(device)
        tab   = batch["tab"].to(device)
        label = batch["label"].to(device)
        logits, aux = model(mri, pet, tab, return_aux=True)
        if criterion:
            loss, _ = criterion(logits, aux, label)
            total_loss += loss.item()
        all_logits.append(logits.cpu())
        all_labels.append(label.cpu())
    logits = torch.cat(all_logits)
    labels = torch.cat(all_labels)
    probs  = F.softmax(logits, dim=-1)[:, 1].numpy()
    preds  = logits.argmax(dim=-1).numpy()
    labels_np = labels.numpy()
    metrics = {
        "loss":     total_loss / len(loader) if criterion else None,
        "acc":      (preds == labels_np).mean(),
        "bal_acc":  balanced_accuracy_score(labels_np, preds),
        "auc":      roc_auc_score(labels_np, probs),
    }
    return metrics, preds, labels_np
# ══════════════════════════════════════════════════════════════════════
#  Training Loop (one fold)
# ══════════════════════════════════════════════════════════════════════
def train_one_fold(
    model, train_loader, val_loader,
    device, epochs, args, fold_idx
):
    criterion  = MultimodalLoss(aux_weight=0.3, gamma=2.0, alpha=0.25, label_smoothing=0.1)
    optimizer  = torch.optim.AdamW(
        model.parameters(), lr=args.lr, weight_decay=args.wd
    )
    total_steps = epochs * len(train_loader)
    scheduler   = CosineWarmupScheduler(
        optimizer, warmup_steps=int(0.05 * total_steps),
        total_steps=total_steps, eta_max=args.lr
    )
    scaler = GradScaler()                         
    best_auc   = 0.0
    best_state = None
    history    = []
    for epoch in range(1, epochs + 1):
        model.train()
        ep_loss = 0.0
        train_preds, train_labels = [], []
        t0 = time.time()
        for batch in train_loader:
            mri   = batch["mri"].to(device)
            pet   = batch["pet"].to(device)
            tab   = batch["tab"].to(device)
            label = batch["label"].to(device)
            optimizer.zero_grad(set_to_none=True)
            with autocast(device_type="cuda"):                         # CHANGED
                logits, aux = model(mri, pet, tab, return_aux=True)
                loss, ldict = criterion(logits, aux, label)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            ep_loss += loss.item()
            train_preds.append(logits.detach().argmax(dim=-1).cpu())
            train_labels.append(label.cpu())
            
        val_metrics, _, _ = evaluate(model, val_loader, device, criterion)
        ep_loss /= len(train_loader)
        train_acc = (torch.cat(train_preds) == torch.cat(train_labels)).float().mean().item()
        if val_metrics["auc"] > best_auc:
            best_auc   = val_metrics["auc"]
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        rec = {
            "fold": fold_idx, "epoch": epoch,
            "train_acc": train_acc,
            "train_loss": ep_loss,
            **{f"val_{k}": v for k, v in val_metrics.items()},
            "time": time.time() - t0,
        }
        history.append(rec)
        print(                                                         # CHANGED
            f"Ep {epoch:03d}/{epochs} | "
            f"train_acc {train_acc*100:.1f}% | "
            f"train_loss {ep_loss:.4f} | "
            f"val_acc {val_metrics['acc']*100:.1f}% | "
            f"val_loss {val_metrics['loss']:.4f} | " 
            f"val_auc {val_metrics['auc']:.4f} | "
            f"val_bal {val_metrics['bal_acc']*100:.1f}% | "
            f"{time.time() - t0:.0f}s",
            flush=True                                                 # CHANGED
        )
    model.load_state_dict(best_state)
    return model, history, best_auc
# ══════════════════════════════════════════════════════════════════════
#  Diagnostic: tabular-only baseline (detects score leakage)
# ══════════════════════════════════════════════════════════════════════
def tabular_baseline(train_ds, val_ds_df, tab_mean, tab_std, tabular_cols):
    from sklearn.linear_model import LogisticRegression
    def get_tab_matrix(df):
        X = df[tabular_cols].values.astype(np.float32)
        return (X - tab_mean) / tab_std
    X_train = get_tab_matrix(train_ds.df)
    y_train = train_ds.df["label"].values
    X_val   = get_tab_matrix(val_ds_df)
    y_val   = val_ds_df["label"].values
    clf = LogisticRegression(max_iter=1000, random_state=42)
    clf.fit(X_train, y_train)
    probs = clf.predict_proba(X_val)[:, 1]
    auc   = roc_auc_score(y_val, probs)
    acc   = (clf.predict(X_val) == y_val).mean()
    print(f"{'='*60}")
    print(f"  TABULAR-ONLY BASELINE (Logistic Regression)")
    print(f"  Val Acc: {acc:.3f} | Val AUC: {auc:.3f}")
    if auc > 0.95:
        print("  ⚠  WARNING: tabular features alone nearly perfectly separate")
        print("     classes. Clinical scores (MMSE/CDRSB/FAQ) are likely")
        print("     direct proxies of the label. Deep model AUC=1.0 is")
        print("     driven by tabular branch, not imaging.")
    print(f"{'='*60}")
    return auc
# ══════════════════════════════════════════════════════════════════════
#  Main: Single train/val split (leak-free)
# ══════════════════════════════════════════════════════════════════════
def main(args):
    seed_everything(args.seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device}")
    full_df = pd.read_csv(args.csv)
    labels  = full_df["label"].values
    train_idx, val_idx = next(
        StratifiedKFold(n_splits=5, shuffle=True, random_state=args.seed)
        .split(np.zeros(len(labels)), labels)
    )
    print(f"Train samples: {len(train_idx)} | Val samples: {len(val_idx)}")
    train_ds = AlzheimerMultimodalDataset(
        csv_path=args.csv, mri_root=args.mri_root,
        pet_root=args.pet_root, augment=True,
    )
    train_ds.df = full_df.iloc[train_idx].reset_index(drop=True)
    train_ds._prepare_tabular(fit_stats=True)
    train_ds.mri_map = train_ds._build_path_map(args.mri_root)
    train_ds.pet_map = train_ds._build_path_map(args.pet_root)
    val_ds = AlzheimerMultimodalDataset(
        csv_path=args.csv, mri_root=args.mri_root,
        pet_root=args.pet_root, augment=False,
    )
    val_ds.df = full_df.iloc[val_idx].reset_index(drop=True)
    val_ds._prepare_tabular(fit_stats=False)
    val_ds.tab_mean = train_ds.tab_mean
    val_ds.tab_std  = train_ds.tab_std
    val_ds.mri_map  = val_ds._build_path_map(args.mri_root)
    val_ds.pet_map  = val_ds._build_path_map(args.pet_root)
    tab_auc = tabular_baseline(
        train_ds, val_ds.df,
        train_ds.tab_mean, train_ds.tab_std, TABULAR_COLS
    )
    train_loader = DataLoader(
        train_ds, batch_size=args.bs, shuffle=True,
        num_workers=0, pin_memory=True
    )
    val_loader = DataLoader(
        val_ds, batch_size=args.bs, shuffle=False,
        num_workers=0, pin_memory=True
    )
    model = AlzheimerMultimodalNet(
        tab_in_dim=len(TABULAR_COLS),
        embed_dim=args.embed_dim,
        fused_dim=args.fused_dim,
        dropout=args.dropout,
    ).to(device)
    print(f"Parameters: {model.count_parameters():,}")
    model, history, best_auc = train_one_fold(
        model, train_loader, val_loader,
        device, args.epochs, args, fold_idx=1
    )
    ckpt_path = Path(args.ckpt_dir) / "best_model1.pt"
    ckpt_path.parent.mkdir(parents=True, exist_ok=True)
    torch.save(model.state_dict(), ckpt_path)
    print(f"Best val AUC : {best_auc:.4f}")
    print(f"Tab baseline : {tab_auc:.4f}")
    print(f"Imaging gain : {best_auc - tab_auc:+.4f}")
    print(f"Checkpoint   : {ckpt_path}")
    history_path = Path(args.ckpt_dir) / "training_history.csv"
    pd.DataFrame(history).to_csv(history_path, index=False)
    print(f"History      : {history_path}")

In [8]:
class Args:
    csv       = "/kaggle/input/datasets/sraddhasigdel/datasetmciad/Multimodalmciad/csv.csv"
    mri_root  = "/kaggle/input/datasets/sraddhasigdel/datasetmciad/Multimodalmciad/MRI"
    pet_root  = "/kaggle/input/datasets/sraddhasigdel/datasetmciad/Multimodalmciad/PET"
    ckpt_dir  = "/kaggle/working/checkpoints"
    folds     = 5
    epochs    = 50
    patience = 15
    bs        = 2        # Kaggle GPU has ~16GB; batch 2 is safe for 96³ volumes
    lr        = 3e-4
    wd        = 1e-4
    dropout   = 0.5
    embed_dim = 128
    fused_dim = 256
    seed      = 42
    tab_in_dim = 10 

args = Args()

In [9]:
main(args)

Device: cuda
Train samples: 318 | Val samples: 80
  TABULAR-ONLY BASELINE (Logistic Regression)
  Val Acc: 0.600 | Val AUC: 0.651
Parameters: 7,756,747
Ep 001/50 | train_acc 52.5% | train_loss 0.1140 | val_acc 55.0% | val_loss 0.0906 | val_auc 0.9144 | val_bal 55.0% | 457s
Ep 002/50 | train_acc 59.4% | train_loss 0.0932 | val_acc 85.0% | val_loss 0.0614 | val_auc 0.9163 | val_bal 85.0% | 128s
Ep 003/50 | train_acc 71.7% | train_loss 0.0895 | val_acc 78.8% | val_loss 0.0842 | val_auc 0.8875 | val_bal 78.8% | 126s
Ep 004/50 | train_acc 70.4% | train_loss 0.0974 | val_acc 81.2% | val_loss 0.0777 | val_auc 0.8925 | val_bal 81.2% | 124s
Ep 005/50 | train_acc 72.3% | train_loss 0.0961 | val_acc 78.8% | val_loss 0.0846 | val_auc 0.8394 | val_bal 78.8% | 126s
Ep 006/50 | train_acc 73.9% | train_loss 0.0971 | val_acc 80.0% | val_loss 0.0957 | val_auc 0.8506 | val_bal 80.0% | 128s
Ep 007/50 | train_acc 69.5% | train_loss 0.0935 | val_acc 80.0% | val_loss 0.0939 | val_auc 0.8181 | val_bal 80.0% |